In [1]:
import numpy as np

# --- Setup ---
# Imagine we have 5 variants, each with a 4-dimensional ESM embedding
# (In reality: hundreds of variants, D=1280)

np.random.seed(42)
n_variants = 5
D = 4

# Variant frequencies (must sum to 1)
f = np.array([0.4, 0.25, 0.15, 0.12, 0.08])
assert np.isclose(f.sum(), 1.0)

# ESM embeddings for each variant: shape (n_variants, D)
# These are continuous vectors, NOT one-hot
Phi = np.array([
    [0.8, 0.1, 0.5, 0.3],   # variant 0
    [0.2, 0.9, 0.1, 0.7],   # variant 1
    [0.6, 0.4, 0.8, 0.2],   # variant 2
    [0.1, 0.7, 0.3, 0.9],   # variant 3
    [0.5, 0.5, 0.6, 0.4],   # variant 4
])  # shape: (5, 4)


# =============================================================================
# SHARED STEP: Mean embedding x(t) — same in both methods
# =============================================================================
# x_d = sum_alpha  f_alpha * phi_alpha[d]
x = f @ Phi   # shape: (D,)  — just a frequency-weighted average of embeddings
print("Mean embedding x(t):")
print(x)
# e.g. x[0] = 0.4*0.8 + 0.25*0.2 + 0.15*0.6 + 0.12*0.1 + 0.08*0.5


# =============================================================================
# METHOD 1: Full covariance  C = E[phi phi^T] - x x^T
# =============================================================================
# Step 1: Weighted second-moment matrix M = sum_alpha f_alpha * phi_alpha phi_alpha^T
# For each variant, compute outer product and accumulate weighted
M = np.zeros((D, D))
for alpha in range(n_variants):
    outer = np.outer(Phi[alpha], Phi[alpha])   # shape: (D, D)
    M += f[alpha] * outer

# Equivalently in one line using einsum:
# M = np.einsum('a,ad,ae->de', f, Phi, Phi)

# Step 2: Subtract mean outer product
x_outer = np.outer(x, x)                       # shape: (D, D)
C_full = M - x_outer

print("\nFull covariance matrix C (D x D):")
print(np.round(C_full, 4))


# =============================================================================
# METHOD 2: Independent-sites approximation
# =============================================================================
# Pretend each dimension d is a binary "allele" with frequency x_d.
# This means:  M ≈ diag(x)   (the expensive sum is replaced by the mean vector)
#
# Diagonal:     C_dd ≈  x_d (1 - x_d)
# Off-diagonal: C_de ≈ -x_d * x_e

C_approx = np.outer(-x, x)                     # fills all entries with -x_d * x_e
np.fill_diagonal(C_approx, x * (1 - x))        # overwrite diagonal with x_d(1-x_d)

print("\nIndependent-sites approximation C (D x D):")
print(np.round(C_approx, 4))


# =============================================================================
# COMPARISON
# =============================================================================
print("\nDifference (full - approx):")
print(np.round(C_full - C_approx, 4))

print("\nDiagonal comparison (variance terms):")
print(f"  Full:   {np.round(np.diag(C_full), 4)}")
print(f"  Approx: {np.round(np.diag(C_approx), 4)}")

# The approximation error on the diagonal is exactly the difference between
# the true weighted second moment E[phi_d^2] and x_d (which would only be
# equal if phi were binary/one-hot).
print("\nWhy they differ on diagonal — true E[phi_d^2] vs x_d:")
E_phi_sq = np.einsum('a,ad->d', f, Phi**2)    # true freq-weighted mean of phi^2
print(f"  E[phi_d^2]: {np.round(E_phi_sq, 4)}")
print(f"  x_d:        {np.round(x, 4)}")
print("  (These would be equal only if embeddings were binary/one-hot)")

Mean embedding x(t):
[0.512 0.449 0.429 0.465]

Full covariance matrix C (D x D):
[[ 0.0791 -0.0885  0.045  -0.0623]
 [-0.0885  0.1077 -0.0529  0.0643]
 [ 0.045  -0.0529  0.0541 -0.0464]
 [-0.0623  0.0643 -0.0464  0.0583]]

Independent-sites approximation C (D x D):
[[ 0.2499 -0.2299 -0.2196 -0.2381]
 [-0.2299  0.2474 -0.1926 -0.2088]
 [-0.2196 -0.1926  0.245  -0.1995]
 [-0.2381 -0.2088 -0.1995  0.2488]]

Difference (full - approx):
[[-0.1708  0.1414  0.2646  0.1758]
 [ 0.1414 -0.1397  0.1397  0.2731]
 [ 0.2646  0.1397 -0.1909  0.1531]
 [ 0.1758  0.2731  0.1531 -0.1905]]

Diagonal comparison (variance terms):
  Full:   [0.0791 0.1077 0.0541 0.0583]
  Approx: [0.2499 0.2474 0.245  0.2488]

Why they differ on diagonal — true E[phi_d^2] vs x_d:
  E[phi_d^2]: [0.3412 0.3093 0.2381 0.2745]
  x_d:        [0.512 0.449 0.429 0.465]
  (These would be equal only if embeddings were binary/one-hot)
